# Libraries and global variables

In [4]:
import wikipediaapi
import pandas as pd
import random
import time
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Define User-Agent (required by Wikipedia)
USER_AGENT = "Ughhhghghh/1.0 (AnEmail@example.com)"

# Initialize Wikipedia API for French articles
wiki_wiki = wikipediaapi.Wikipedia(
    user_agent=USER_AGENT, language='fr', extract_format=wikipediaapi.ExtractFormat.WIKI
)

# Categories to explore
categories = [
    "Histoire", "Science", "Culture", "Technologie", "Économie", "Politique",
    "Géographie", "Littérature", "Sport", "Musique"
]

# Minimum number of articles per category
min_articles_per_category = 1500
# Minimum number of characters per article
min_characters = 2000
# Number of parallel threads for faster fetching
num_threads = 8  # Reduce to avoid rate limits
# Max retries for failed API calls
max_retries = 5
# Initial backoff time in seconds
initial_backoff = 1

# Data storage
data = []
all_articles = []


# Fetch article titles


In [2]:
def get_articles_from_category(category, min_articles=500, max_level=10):
    """ Recursively fetch articles from a category and its subcategories, stopping early if min_articles is reached. """
    articles = set()
    visited_categories = set()

    def fetch_category_members(category_page, level=0):
        if level > max_level or category_page.title in visited_categories or len(articles) >= min_articles:
            return  # Stop recursion early if we have enough articles
        visited_categories.add(category_page.title)

        for page in category_page.categorymembers.values():
            if len(articles) >= min_articles:  # Stop early if enough articles are collected
                return
            
            # If it's an article, add it
            if page.ns == wikipediaapi.Namespace.MAIN:
                articles.add(page.title)
            # If it's a subcategory, recurse into it
            elif page.ns == wikipediaapi.Namespace.CATEGORY:
                fetch_category_members(page, level + 1)

    # Start recursive fetching from the main category
    category_page = wiki_wiki.page(f"Catégorie:{category}")
    if not category_page.exists():
        print(f"⚠️ Category '{category}' does not exist.")
        return set()

    fetch_category_members(category_page)

    # Ensure minimum number of articles
    article_list = list(articles)
    random.shuffle(article_list)

    print(f"✅ Found {len(article_list)} articles in '{category}' (Needed: {min_articles})")

    return article_list  # Return whatever was collected (should be at least `min_articles` now)


In [3]:
# **Step 1A: Collect all article titles with progress tracking**
print("\n📌 Fetching article titles...\n")

for category in tqdm(categories, desc="Processing Categories", unit="category"):
    category_articles = get_articles_from_category(category, min_articles=min_articles_per_category)
    all_articles.extend([(category, title) for title in category_articles])

# Shuffle articles for randomness
random.shuffle(all_articles)
print(f"\n✅ Total articles collected: {len(all_articles)}")


📌 Fetching article titles...



Processing Categories:  10%|█         | 1/10 [00:03<00:27,  3.08s/category]

✅ Found 1500 articles in 'Histoire' (Needed: 1500)


Processing Categories:  20%|██        | 2/10 [00:04<00:16,  2.03s/category]

✅ Found 1500 articles in 'Science' (Needed: 1500)


...

✅ Found 1500 articles in 'Littérature' (Needed: 1500)


Processing Categories:  90%|█████████ | 9/10 [01:30<00:08,  8.96s/category]

✅ Found 1500 articles in 'Sport' (Needed: 1500)


Processing Categories: 100%|██████████| 10/10 [01:35<00:00,  9.57s/category]

✅ Found 1500 articles in 'Musique' (Needed: 1500)

✅ Total articles collected: 15000


In [4]:
print(all_articles)

[('Musique', 'The Cambridge Buskers'), ('Culture', 'Néférourê'), ('Littérature', 'Joseph et ses frères'), ('Littérature', 'Les Deux Frères (conte de Grimm)'), ('Littérature', 'Hans Robert Jauss'), ('Littérature', 'Walter Muschg'), ('Économie', 'Industrie du sexe'), ('Sport', 'Saber Desfarges'), ('Littérature', 'Grand Catéchisme de Luther'), ('Musique', 'Accord de quinte diminuée'), ('Géographie', 'Cap de Trafalgar'), ('Technologie', 'Brosse de toilettes'), ('Littérature', 'Littérature finlandaise'), ('Histoire', 'Connétable de Normandie'), ('Musique', 'Chant liturgique'), ('Politique', 'DONUT'), ('Science', 'KIFC3'), ('Culture', 'Antigone (Phthie)'), ('Politique', 'Chan Santokhi'), ('Économie', 'Horeca'), ('Musique', 'Samba-choro'), ('Géographie', 'Géographie économique'), ('Sport', 'Association olympique britannique'), ('Littérature', 'Gatta Cenerentola'), ('Science', 'Entérocœlie'), ('Science', 'Christian Bonah'), ('Politique', 'Massacres de Río Negro'), ('Musique', 'Marco Beasley'),

In [8]:
# Save all_articles to a file
with open('all_articles.json', 'w', encoding='utf-8') as f:
    json.dump(all_articles, f, ensure_ascii=False, indent=4)

# Read all_articles from the file
with open('all_articles.json', 'r', encoding='utf-8') as f:
    all_articles = json.load(f)

all_articles = [tuple(article) for article in all_articles]
print(f"✅ Loaded {len(all_articles)} articles from file.")

✅ Loaded 15000 articles from file.


# Fetch Full Text & Summary for Each Article

In [29]:
import re

# List of section titles to exclude (case-insensitive)
EXCLUDED_SECTIONS = {
    "Notes et Références", "Notes", "Références", "Voir aussi",
    "Liens externes", "Bibliographie", "Annexes", "Sources",
    "Articles connexes", "Publications", "Autres projets",
    "Portail", "Iconographie", "Filmographie"
}

def clean_wikipedia_sections(page):
    """Filter out empty or irrelevant sections while preserving useful content."""
    def extract_sections(sections):
        cleaned_text = []
        for section in sections:
            title = section.title.strip()

            # Skip sections that match exclusion list exactly (case insensitive)
            if title.lower() in {s.lower() for s in EXCLUDED_SECTIONS}:
                continue
            
            text = section.text.strip()
            
            # Ensure it's not an empty section or just a table
            if text and not re.fullmatch(r'(\s*{\|.*?\|}\s*)+', text, re.DOTALL):
                cleaned_text.append(f"{title}\n{text}")

            # Recursively process subsections
            cleaned_text.extend(extract_sections(section.sections))
        
        return cleaned_text

    return page.summary + "\n\n" + "\n\n".join(extract_sections(page.sections))

def fetch_article_content(article_tuple):
    """Fetch full text & summary for a Wikipedia article with retries, filtering empty sections."""
    category, title = article_tuple
    backoff = initial_backoff

    for attempt in range(max_retries):
        try:
            page = wiki_wiki.page(title)

            if page.exists():
                summary = page.summary
                filtered_text = clean_wikipedia_sections(page)

                # Check minimum character length
                if len(filtered_text) >= min_characters:
                    return {
                        "Category": category,
                        "Title": title,
                        "Summary": summary,
                        "Text": filtered_text
                    }

            return None  # Skip articles that don’t meet criteria

        except json.JSONDecodeError:
            print(f"⚠️ JSONDecodeError for {title}. Retrying ({attempt + 1}/{max_retries})...")
            time.sleep(backoff)
            backoff *= 2  # Exponential backoff

        except Exception as e:
            print(f"⚠️ Unexpected error for {title}: {e}. Retrying ({attempt + 1}/{max_retries})...")
            time.sleep(backoff)
            backoff *= 2  # Exponential backoff

    print(f"❌ Failed to fetch {title} after {max_retries} retries.")
    return None

In [31]:
print("\n📌 Fetching article content (parallel requests)...\n")

with ThreadPoolExecutor(max_workers=num_threads) as executor:
    futures = {executor.submit(fetch_article_content, article): article for article in all_articles}
    
    # Track progress
    for future in tqdm(as_completed(futures), total=len(all_articles), desc="Fetching Articles", unit="article"):
        result = future.result()
        if result:
            data.append(result)  # Store successful results


📌 Fetching article content (parallel requests)...



Fetching Articles: 100%|██████████| 15000/15000 [10:22<00:00, 24.09article/s]


# Save to csv

In [32]:
# Convert to DataFrame and save as CSV
df = pd.DataFrame(data)
csv_filename = "french_wikipedia_articles.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8")

print(f"\n✅ Dataset saved: {len(df)} articles collected! File: {csv_filename}")



✅ Dataset saved: 12694 articles collected! File: french_wikipedia_articles.csv


In [35]:
# Sample 10 random articles from data
sampled_articles = random.sample(data, 10)

for article in sampled_articles[:10]:
    title = article['Title']
    summary_length = article['Summary']
    text_length = article['Text']
    print(f"\n📚 {title}\n🔖 Summary: {summary_length}\n📝 Text: \n{text_length}")
    print("\n" + "-"*50)


📚 Dilma Rousseff
🔖 Summary: Dilma Vana Rousseff (/ˈd͡ʒiwmɐ ˈvɐnɐ ʁuˈsɛf(i)/), née le 14 décembre 1947 à Belo Horizonte (Brésil), est une économiste et femme d'État brésilienne. Membre du Parti des travailleurs (PT), elle est présidente de la république fédérative du Brésil du 1er janvier 2011 au 31 août 2016.
Ministre des Mines et de l'Énergie de 2003 à 2005 puis ministre de la Maison civile du président Luiz Inácio Lula da Silva à partir de 2005, elle est la candidate du PT à l'élection présidentielle de 2010, qu'elle remporte au second tour face à José Serra.
Première femme à exercer la fonction de chef de l'État au Brésil, elle est réélue de justesse en 2014, face à Aécio Neves. Sa présidence est marquée par un déclin de l'économie brésilienne et par des scandales de corruption touchant une grande partie de la classe politique.
Elle est destituée pour maquillage des comptes publics par le Sénat le 31 août 2016, au terme d'une procédure controversée. Elle échoue à devenir sénatrice 

In [30]:
article = "Climat du Sahara"
page = wiki_wiki.page(article)
print(page.text)
print("_"*50)
print(clean_wikipedia_sections(page))

Le climat du Sahara, ou plutôt les climats du Sahara, connu pour être le plus grand désert chaud au monde, possède des caractéristiques très similaires mais parfois bien distinctes selon les différentes régions dans ce « désert subtropical » vaste, aride et absolu, le climat n'y est pas uniforme sur les 9 millions de km² de désert. 
Dans cette partie d'Afrique du Nord règne un climat désertique chaud (Classification de Köppen BWh) caractérisé par une sécheresse extrême avec des précipitations rares et faibles, de très fortes chaleurs avec des températures excessivement élevées pendant une période plus ou moins longue, une très forte irradiation solaire avec une durée d'ensoleillement record dans une grande partie ainsi qu'une très faible humidité et une grande siccité de l'atmosphère et par des vents réguliers, calmes et rarement violents. Ces caractéristiques climatiques du Sahara ne sont qu'un résumé global de la situation bien plus complexe en réalité mais il est important d'y appor